# 📊 Notebook 01 — Exploratory Data Analysis (EDA)
**Sentiment Analysis API Project**

This notebook covers:
- Loading SST-2, IMDB, Amazon datasets
- Class distribution analysis
- Text length statistics
- Vocabulary analysis
- Word clouds and frequency plots
- Data quality checks

In [ ]:
import os, re, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from collections import Counter
from datasets import load_from_disk, load_dataset
from transformers import AutoTokenizer
from tqdm.notebook import tqdm

warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.facecolor': '#0d1117', 'axes.facecolor': '#161b22',
                     'text.color': '#c9d1d9', 'axes.labelcolor': '#c9d1d9',
                     'xtick.color': '#8b949e', 'ytick.color': '#8b949e',
                     'axes.edgecolor': '#30363d', 'grid.color': '#21262d',
                     'figure.figsize': (14, 6)})

TOKENIZER_NAME = 'distilbert-base-uncased'
DATA_DIR = '../data/raw'
print('✔ Imports OK')

In [ ]:
# ── Load SST-2 ─────────────────────────────────────────────────────
try:
    sst2 = load_from_disk(f'{DATA_DIR}/sst2')
    print(f'SST-2 loaded from disk: {sst2}')
except:
    print('Downloading SST-2...')
    sst2 = load_dataset('sst2')

train_df = pd.DataFrame(sst2['train'])
val_df   = pd.DataFrame(sst2['validation'])
print(f'Train: {len(train_df):,} | Val: {len(val_df):,}')
train_df.head()

In [ ]:
# ── Class Distribution ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

COLORS = ['#ff4757', '#00f5ff', '#00ff88']

for ax, df, title in zip(axes, [train_df, val_df], ['Train Split', 'Validation Split']):
    counts = df['label'].value_counts().sort_index()
    labels = ['NEGATIVE (0)', 'POSITIVE (1)']
    bars = ax.bar(labels, counts.values, color=COLORS[:2], width=0.5, edgecolor='none')
    for bar, val in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
                f'{val:,}', ha='center', va='bottom', color='#c9d1d9', fontsize=11)
    ax.set_title(f'Class Distribution — {title}', fontsize=13, pad=12)
    ax.set_ylabel('Sample Count')
    ax.grid(axis='y', alpha=0.3)
    ax.set_ylim(0, counts.max() * 1.15)

plt.suptitle('SST-2 Class Balance Analysis', fontsize=15, y=1.02)
plt.tight_layout()
plt.savefig('../data/processed/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Text Length Analysis ───────────────────────────────────────────
train_df['char_len']  = train_df['sentence'].str.len()
train_df['word_len']  = train_df['sentence'].str.split().str.len()

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME)
print('Tokenizing (this takes a moment)...')
train_df['token_len'] = train_df['sentence'].apply(
    lambda x: len(tokenizer.encode(x, truncation=False))
)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, col, label, color in zip(
    axes,
    ['char_len', 'word_len', 'token_len'],
    ['Character Length', 'Word Count', 'Token Count'],
    ['#00f5ff', '#b400ff', '#00ff88']
):
    ax.hist(train_df[col], bins=50, color=color, alpha=0.85, edgecolor='none')
    ax.axvline(train_df[col].median(), color='#ff4757', ls='--', lw=1.5, label=f'Median={train_df[col].median():.0f}')
    ax.axvline(train_df[col].quantile(0.95), color='#ffb347', ls=':', lw=1.5, label=f'P95={train_df[col].quantile(0.95):.0f}')
    ax.set_xlabel(label); ax.set_ylabel('Frequency')
    ax.set_title(f'{label} Distribution')
    ax.legend(fontsize=9); ax.grid(alpha=0.3)

plt.suptitle('Text Length Statistics — SST-2 Train', fontsize=14)
plt.tight_layout()
plt.savefig('../data/processed/text_lengths.png', dpi=150, bbox_inches='tight')
plt.show()

print(train_df[['char_len','word_len','token_len']].describe().round(1))

In [ ]:
# ── Token Length by Class ──────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))

for label_id, label_name, color in [(0, 'NEGATIVE', '#ff4757'), (1, 'POSITIVE', '#00ff88')]:
    subset = train_df[train_df['label'] == label_id]['token_len']
    ax.hist(subset, bins=40, alpha=0.65, color=color, label=f'{label_name} (n={len(subset):,})', edgecolor='none')

ax.axvline(128, color='#ffb347', ls='--', lw=2, label='Max length = 128')
ax.set_xlabel('Token Count'); ax.set_ylabel('Frequency')
ax.set_title('Token Length Distribution by Sentiment Class')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

trunc_pct = (train_df['token_len'] > 128).mean() * 100
print(f'Samples exceeding 128 tokens: {trunc_pct:.2f}% — these will be truncated during training')

In [ ]:
# ── Top 30 Word Frequencies ───────────────────────────────────────
import re
STOPWORDS = {'the','a','an','is','it','in','of','and','to','that','this','was','with','for',
             'are','be','as','at','has','have','its','on','s','not','but','by','or','from'}

def get_words(texts):
    all_words = []
    for t in texts:
        words = re.findall(r'\b[a-z]+\b', t.lower())
        all_words.extend([w for w in words if w not in STOPWORDS and len(w) > 2])
    return Counter(all_words)

pos_words = get_words(train_df[train_df.label==1]['sentence'])
neg_words = get_words(train_df[train_df.label==0]['sentence'])

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
for ax, counter, title, color in [
    (axes[0], pos_words, 'Top 30 — POSITIVE', '#00ff88'),
    (axes[1], neg_words, 'Top 30 — NEGATIVE', '#ff4757'),
]:
    top = counter.most_common(30)
    words, counts = zip(*top)
    ax.barh(list(reversed(words)), list(reversed(counts)), color=color, alpha=0.85, edgecolor='none')
    ax.set_xlabel('Frequency'); ax.set_title(title, fontsize=13)
    ax.grid(axis='x', alpha=0.3)

plt.suptitle('Most Frequent Words by Sentiment', fontsize=14)
plt.tight_layout()
plt.savefig('../data/processed/word_frequencies.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Data Quality Report ───────────────────────────────────────────
print('=' * 55)
print('         DATA QUALITY REPORT — SST-2')
print('=' * 55)

# Null check
nulls = train_df.isnull().sum()
print(f'\n  Null values:\n{nulls}\n')

# Duplicates
dups = train_df['sentence'].duplicated().sum()
print(f'  Duplicate sentences: {dups} ({dups/len(train_df)*100:.2f}%)')

# Empty/short texts
short = (train_df['word_len'] < 3).sum()
print(f'  Texts < 3 words:    {short}')

# Class balance
balance = train_df['label'].value_counts(normalize=True) * 100
print(f'\n  Class balance:')
for label, pct in balance.items():
    print(f'    Label {label}: {pct:.1f}%')

print(f'\n  Avg token length: {train_df.token_len.mean():.1f} ± {train_df.token_len.std():.1f}')
print(f'  P99 token length: {train_df.token_len.quantile(0.99):.0f}')
print('=' * 55)

# Save summary
import json
os.makedirs('../data/processed', exist_ok=True)
summary = {
    'total_train': len(train_df),
    'total_val': len(val_df),
    'class_balance': {str(k): round(v/100, 4) for k, v in balance.items()},
    'avg_tokens': round(train_df.token_len.mean(), 1),
    'p99_tokens': int(train_df.token_len.quantile(0.99)),
    'duplicates': int(dups),
}
with open('../data/processed/eda_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print('\n✔ EDA summary saved to data/processed/eda_summary.json')